### Load data 

In [1]:
# Load CSEC2017 synthetic + KD2017 datasets 
from datasets import load_from_disk
import datasets
datasets.disable_caching()
import sys
sys.path.append("/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu") 

KAs = {"0": "miscellaneous (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

# edited after IEEE Access reviewer comments 
KAs = {
    0: {
        "name": "Miscellaneous",
        "definition": "Computer Science, Business & Law, Communication & Networking, Information Technology, Cyberspace Practice, Pedagogy, Intelligence"
    },
    1: {
        "name": "Data Security",
        "definition": "Basic cryptography concepts, Digital forensics, End-to-end secure communications, Data integrity & authentication, Information storage security"
    },
    2: {
        "name": "Software Security",
        "definition": "Fundamental design principles, Security requirements, Implementation issues, Static & dynamic testing, Configuring & patching, Ethics"
    },
    3: {
        "name": "Component Security",
        "definition": "Vulnerabilities of system components, Component lifecycle, Secure component design principles, Supply chain management security, Security testing, Reverse engineering"
    },
    4: {
        "name": "Connection Security",
        "definition": "Systems, architecture, models, standards, Physical component interfaces, Software component interfaces, Connection attacks, Transmission attacks"
    },
    5: {
        "name": "System Security",
        "definition": "Holistic approach, Security policy, Authentication, Access control, Monitoring, Recovery, Testing, Documentation"
    },
    6: {
        "name": "Human Security",
        "definition": "Identity management, Social engineering, Awareness & understanding, Social behavioral privacy & security, Personal data privacy & security"
    },
    7: {
        "name": "Organizational Security",
        "definition": "Risk management, Governance & policy, Laws, ethics, & compliance, Strategy & planning"
    },
    8: {
        "name": "Societal Security",
        "definition": "Cybercrime, Cyber law, Cyber ethics, Cyber policy, Privacy"
    }
}

from utils.load_data import preprocess
# KDs 
KD_dataset = datasets.load_dataset("csv",data_files={"train": "/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/train_data.csv"}, split='train')
KD_dataset = KD_dataset.remove_columns('KSAT ID')
KD_dataset = KD_dataset.map(preprocess)
print(KD_dataset)

from utils.load_data import preprocess_csec, preprocess_csec8
# CSEC2017 specific! CHANGED: train_CSEC2017b to train_CSEC2017c to include class 0 
csec_dataset = datasets.load_dataset("csv",data_files={"train": "/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/train_CSEC2017c.csv"}, split='train')
csec_dataset = csec_dataset.remove_columns('Statement Description')
csec_dataset = csec_dataset.remove_columns('label')
csec_dataset = csec_dataset.select_columns(['topics','0','1','2','3','4','5','6','7','8'])
csec_dataset = csec_dataset.map(preprocess_csec)
print(csec_dataset)

Map:   0%|          | 0/576 [00:00<?, ? examples/s]

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 576
})


Map:   0%|          | 0/2143 [00:00<?, ? examples/s]

Dataset({
    features: ['topics', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2143
})


In [2]:
# clean datasets and convert to pandas 
import pandas as pd 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from utils.load_data import clean_text

feature_type = 'tf-idf' 

pandas_dataCSEC = pd.DataFrame({'topics': csec_dataset['topics'], 'labels': csec_dataset['labels']})
pandas_dataCSEC['cleaned'] = pandas_dataCSEC['topics'].apply(clean_text)

# remove knowledge of (indiscriminative)
def remove_knowledge_of(example):
    example = example[13:]
    return example

pandas_dataKD = pd.DataFrame({'topics': KD_dataset['Statement Description'], 'labels': KD_dataset['labels']})
pandas_dataKD['cleaned'] = pandas_dataKD['topics'].apply(clean_text).apply(remove_knowledge_of)
# merge fine-tuning datasets 
mergeds = pd.concat([pd.DataFrame({'topics': pandas_dataCSEC['cleaned'], 'labels': pandas_dataCSEC['labels']}), 
                     pd.DataFrame({'topics': pandas_dataKD['cleaned'], 'labels': pandas_dataKD['labels']})]).reset_index() # for
# convert labels to int 
mergeds['labels'] = mergeds['labels'].apply(lambda x: [int(i) for i in x])

In [3]:
# Load Qwen2-7B-Instruct model. 
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

import json
from tqdm import tqdm
import re 

def label_KD_prompt(knowledge, KAs):
        """
        Formats a single question+answers into a list of message dictionaries for the pipeline.
        """
        options_str = ', '.join([f"{key}) {value}" for key, value in KAs.items()])
        instructions = (
            "You are a helpful AI assistant.\n"
            "Instructions:\n"
            "a. Carefully read the knowledge statement.\n"
            "b. Choose one or more of the following (0, 1, 2, 3, 4, 5, 6, 7, 8).\n"
            "c. Do NOT include any explanation or additional text in the response.\n"
           # "d. Always return the answer in this format: 'answer'. "
            #"For example, if the correct answers are 0 and 1, then return 0,1.\n\n"
        )
    
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": 
            f"#Question: Classify the following statement {knowledge} into one or multiple of the following knowledge areas: \nOptions: {options_str}"}
        ]
        return messages

def submit_message_LLM(model, messages): 
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

def process_label(label): 
    output = [int(s) for s in re.findall(r'\d+', label)]
    output = str(output)
    output = output.replace(' ','')[1:-1]
    output = [int(num) for num in output.split(',')]
    return output 

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [4]:
print(mergeds['topics'][1])

sender authentication


In [5]:
num_classes = 9 

def Zero_Shot_sample(KD): 
    query = label_KD_prompt(KD, KAs)
    predicted_label = process_label(submit_message_LLM(model2, query))
    #print('actual label: ',mergeds['labels'][1])
    one_hot =[1 if j in predicted_label else 0 for j in range(num_classes)]
    #print('predicted label: ',one_hot)
    return one_hot

In [6]:
import numpy as np
num_classes = 9 
KD = mergeds['topics'][1]
query = label_KD_prompt(KD, KAs)
predicted_label = process_label(submit_message_LLM(model2, query))

one_hot =[1 if j in predicted_label else 0 for j in range(num_classes)]

print('predicted label: ',one_hot)
print('actual label: ',mergeds['labels'][1])

predicted label:  [0, 1, 0, 0, 0, 1, 0, 0, 0]
actual label:  [0, 1, 0, 0, 0, 0, 0, 0, 0]


In [7]:
# To do: Train 5 times and average the resulting metrics. 
# To do: 5 different random seeds as well. 
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score 
n_repeats = 1 # number of seeds 
n_splits = 10 # number of k splits 
precisions = []
recalls = [] 
f1s = [] 
accs = []
for _ in range(n_repeats): 
    kf = KFold(n_splits=n_splits,  shuffle=True) 
    precision = 0 
    recall = 0 
    f1 = 0 
    acc = 0 
    for i, (train, test) in enumerate(kf.split(mergeds)):  # split into k-folds
        print('split ',i)
        # extract train and test dataset for k-folds cross-validation
        train_ds = mergeds.loc[train] 
        test_ds = mergeds.loc[test] 
        # do zero-shot labelling on the test dataset 
        predicted_labels = test_ds['topics'].apply(Zero_Shot_sample)
        # compute metrics on the validation dataset 
        report = classification_report(test_ds['labels'].tolist(),predicted_labels.tolist()
                                       ,output_dict=True, zero_division=0)
        accuracy = accuracy_score(test_ds['labels'].tolist(),predicted_labels.tolist())
        # gather and normalize metrics by the length of the test dataset 
        acc += (accuracy * len(test_ds)) / len(mergeds)
        precision += (report['macro avg']['precision'] * len(test_ds)) / len(mergeds)
        recall += (report['macro avg']['recall'] * len(test_ds)) / len(mergeds)
        f1 += (report['macro avg']['f1-score'] * len(test_ds)) / len(mergeds)
    
    precisions.append(precision) 
    recalls.append(recall) 
    f1s.append(f1)
    accs.append(acc) 
# print the results
print('precision: ',np.mean(precisions))
print('precision std: ',np.std(precisions))
print('recall: ',np.mean(recalls))
print('recall std: ',np.std(recalls))
print('f1: ',np.mean(f1s))
print('f1 std: ',np.std(f1s))
print('accuracy: ',np.mean(accs))
print('accuracy std: ',np.std(accs))

split  0
split  1
split  2
split  3
split  4
split  5
split  6
split  7
split  8
split  9
precision:  0.35366839349350176
precision std:  0.0
recall:  0.45884310592353733
recall std:  0.0
f1:  0.37716757948208207
f1 std:  0.0
accuracy:  0.21184258918720114
accuracy std:  0.0


In [8]:
np.savetxt('output/precision_ZS_Qwen.txt', precisions, delimiter=',')
np.savetxt('output/recall_ZS_Qwen.txt', recalls, delimiter=',')
np.savetxt('output/f1_ZS_Qwen.txt', f1s, delimiter=',')
np.savetxt('output/accuracy_ZS_Qwen.txt', accs, delimiter=',')

In [9]:
### Inserted after IEEE Reviewer comments 
#1. Load independent test set 
import pandas as pd 
test_df = pd.read_excel('/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/Topics_Annotation.xlsx')
print(test_df.head())
# label the test set using ChatGPT-4o-mini 
pred = test_df['Topic'].apply(Zero_Shot_sample)
print(pred)

                                Topic Annotator 1 Annotator 2  Annotator 3   \
0                     Ethical Hacking         1,6          6,8        2,5,6   
1  Network and vulnerability scanning           4      3,4,5,7          1,4   
2       Exploit development platforms         2,7            3          2,3   
3                 Command and control           0            2        2,3,6   
4                   Password cracking           1          5,6        1,2,3   

               Sara  Valtteri         Paul CuricuLLM (one specific run)  
0  0,1,2,3,4,5,6,7,8    2,7,8  3,2,4,5,6,0                            8  
1                4,5        5          4,0                            4  
2                  2        2        2,3,4                            2  
3                  0        7      0,2,4,5                            0  
4                0,1        5        1,2,6                            6  
0     [0, 1, 0, 0, 0, 1, 0, 0, 1]
1     [0, 1, 0, 0, 0, 1, 0, 0, 0]
2     [0, 1, 

In [10]:
pred.to_csv("output/pred_ZS_Qwen.csv", index=False) 
pred2 = pd.read_csv("output/pred_ZS_Qwen.csv")
pred2.head()

,Topic
0,"[0, 1, 0, 0, 0, 1, 0, 0, 1]"
1,"[0, 1, 0, 0, 0, 1, 0, 0, 0]"
2,"[0, 1, 1, 0, 0, 0, 0, 0, 0]"
3,"[1, 0, 0, 0, 0, 1, 0, 0, 0]"
4,"[0, 1, 0, 0, 0, 1, 0, 0, 1]"


split  0
split  1
split  2
split  3
split  4
split  5
split  6
split  7
split  8
split  9
precision:  0.28896467876425513
precision std:  0.0
recall:  0.43815864004288174
recall std:  0.0
f1:  0.3155595275350578
f1 std:  0.0
accuracy:  0.1658698050753954
accuracy std:  0.0


In [23]:
print(predicted_labels)

29      [0, 1, 0, 0, 0, 0, 0, 0, 0]
32      [1, 0, 0, 0, 0, 0, 0, 0, 0]
43      [0, 1, 0, 0, 0, 0, 0, 0, 0]
44      [0, 1, 0, 0, 0, 0, 0, 0, 1]
45      [0, 1, 1, 0, 0, 0, 0, 0, 0]
                   ...             
2657    [1, 0, 0, 0, 0, 0, 0, 0, 0]
2662    [0, 1, 0, 0, 0, 1, 0, 1, 0]
2664    [1, 0, 0, 0, 0, 0, 0, 0, 0]
2682    [0, 0, 0, 0, 0, 0, 0, 1, 0]
2692    [0, 0, 0, 0, 0, 0, 0, 1, 0]
Name: topics, Length: 272, dtype: object


In [24]:
report = classification_report(test_ds['labels'].tolist(),predicted_labels.tolist()
                                       ,output_dict=True, zero_division=0)
print(report)

{'0': {'precision': 0.1676300578034682, 'recall': 0.7631578947368421, 'f1-score': 0.27488151658767773, 'support': 38.0}, '1': {'precision': 0.35714285714285715, 'recall': 0.6862745098039216, 'f1-score': 0.4697986577181208, 'support': 51.0}, '2': {'precision': 0.35294117647058826, 'recall': 0.46153846153846156, 'f1-score': 0.4000000000000001, 'support': 39.0}, '3': {'precision': 0.25, 'recall': 0.36363636363636365, 'f1-score': 0.2962962962962963, 'support': 22.0}, '4': {'precision': 0.46153846153846156, 'recall': 0.3, 'f1-score': 0.3636363636363637, 'support': 40.0}, '5': {'precision': 0.18478260869565216, 'recall': 0.5483870967741935, 'f1-score': 0.2764227642276423, 'support': 31.0}, '6': {'precision': 0.36666666666666664, 'recall': 0.34375, 'f1-score': 0.3548387096774193, 'support': 32.0}, '7': {'precision': 0.3918918918918919, 'recall': 0.35365853658536583, 'f1-score': 0.37179487179487175, 'support': 82.0}, '8': {'precision': 0.2608695652173913, 'recall': 0.16666666666666666, 'f1-sco